# CS-MON : Fine-tuned VMamba-Tiny, ordinal learning and evaluation

### Run manually in your existing Windows VS Code window

The Ubuntu GPU environment is installed. Connect this notebook through **Select Kernel ? Select Another Kernel ? Existing Jupyter Server**, enter the authenticated localhost URL supplied in chat, and select **CS-MON GPU (WSL)**. This uses Ubuntu without opening another VS Code window. The server must remain running; after a Windows/WSL restart it must be started again.

Run **01_preprocessing.ipynb ? Run All** and wait for completion. Then run **02_training_evaluation.ipynb ? Run All**. Training currently uses **quick mode** (one seed, CS-MON and four baselines); full multi-seed training and ablations require changing the configuration. Do not run both notebooks simultaneously. The GPU preflight runs automatically before training.

Run `01_preprocessing.ipynb` first. All definitions, training and analysis live here. Outputs are computed from real experiments; this source notebook contains no fabricated results. Quick mode is preliminary. Full mode predeclares comparisons and seeds before final test evaluation. Baselines use exactly the same split, RGB observations, transforms, ImageNet normalization and cumulative ordinal objective.

**Dependencies:** Python 3.10/3.11, PyTorch + matching torchvision, numpy, pandas, Pillow, matplotlib, seaborn, scikit-learn, scikit-image, tqdm, kornia, timm, einops, fvcore. Install packages in the notebook kernel environment, keeping caches outside the project. Windows Jupyter uses `num_workers=0` because these Dataset classes are defined in notebook cells.

**Official VMamba sources:** [repository and installation](https://github.com/MzeroMiko/VMamba), [self-contained implementation](https://github.com/MzeroMiko/VMamba/blob/main/vmamba.py), [Tiny s1l8 configuration](https://github.com/MzeroMiko/VMamba/blob/main/classification/configs/vssm/vmambav2v_tiny_224.yaml), [ImageNet checkpoint](https://github.com/MzeroMiko/VMamba/releases/download/%23v2cls/vssm1_tiny_0230s_ckpt_epoch_264.pth). We use VSSM with SS2D, depths [2,2,8,2], state size 1, SSM ratio 1, LN2D, v2 patch embedding, v3 downsampling and v05_noz scan. Load the entire original 1000-class model strictly before replacing only its classification head with Identity; the retained norm/pooling produce 768-dimensional features.

**Practical GPU setup:** upstream documents Linux Python 3.10, Torch 2.2 / torchvision 0.17, numpy 1.24.4, timm 0.4.12 and its matching `mamba-ssm` 2.2.4 CUDA wheel; its README links the exact wheel. Use a coherent Torch/torchvision/CUDA/Python/ABI combination. Triton also requires the CUDA toolkit's `ptxas` executable; if it is not discoverable, set `TRITON_PTXAS_PATH` to its full path. On Windows, WSL2/Linux with CUDA is the smallest practical fast-training route. Native Windows can run the official PyTorch reference scan for explicit diagnostic use, but it is slow and memory intensive. No approximate substitute is used. The driver-reported CUDA version alone does not prove extension compatibility. A real forward/backward preflight below is mandatory.

Labels are supplied ordinal intensity categories, whose preparation/reference provenance must be documented. Missing physical sample/batch/session metadata limits independence claims. Heatmaps describe image decisions, not chemical identification or safety.

## Reproducible Ubuntu 22.04 GPU environment

Run these terminal commands inside Ubuntu after WSL installation and any required Windows restart. Ubuntu 22.04 supplies Python 3.10. This installs the CUDA 12.1 PyTorch build and the official Mamba wheel matching Python 3.10, PyTorch 2.2 and the **FALSE** C++11 ABI. The upstream README's TRUE-ABI example must not be used with the FALSE-ABI PyTorch wheel. A CUDA-capable Windows driver must already be available to WSL; do not install a Linux display driver inside WSL.

```bash
sudo apt-get update
sudo apt-get install -y python3.10-venv python3-pip build-essential
python3.10 -m venv ~/.venvs/csmon-gpu
source ~/.venvs/csmon-gpu/bin/activate
python -m pip install --upgrade pip
python -m pip install torch==2.2.2 torchvision==0.17.2 --index-url https://download.pytorch.org/whl/cu121
python -m pip install numpy==1.24.4 pandas==2.2.3 scipy==1.10.1 scikit-image==0.21.0 scikit-learn==1.3.2 matplotlib==3.8.4 seaborn==0.13.2 pillow tqdm ipykernel jupyterlab ipywidgets kornia==0.7.2 timm==0.4.12 einops fvcore ninja packaging transformers==4.40.2 setuptools==80.9.0
python -m pip install --no-deps 'https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4%2Bcu12torch2.2cxx11abiFALSE-cp310-cp310-linux_x86_64.whl'
python -m ipykernel install --user --name csmon-gpu --display-name 'CS-MON GPU (WSL)'
python -c "import torch, selective_scan_cuda, triton; print(torch.__version__, torch.version.cuda, torch.cuda.is_available()); assert torch.cuda.is_available(); assert not torch._C._GLIBCXX_USE_CXX11_ABI; print(torch.cuda.get_device_name(0))"
```

Use the existing-window Jupyter connection described above, or open the project in VS Code connected to Ubuntu/WSL and choose **CS-MON GPU (WSL)** as this notebook's kernel. The WSL project path is `/mnt/d/SLIIT (4th Year)/RP/CS-MON`. Keep quick mode and `allow_slow_reference_scan=False`. Run All executes the mandatory GPU preflight before any training and stops if that check fails. Extension import alone does not verify the actual CUDA kernels on this GPU. The environment below has now been installed and validated in this workspace.

### Verified setup: ready for Run All

WSL2 and Ubuntu 22.04.5 are installed and operational. The registered **CS-MON GPU (WSL)** kernel uses `/home/csmon/.venvs/csmon-gpu/bin/python` under the Linux account `csmon`. Installed and checked: Python 3.10.12, PyTorch 2.2.2+cu121, torchvision 0.17.2+cu121, Triton 2.2.0, official Mamba selective scan 2.2.4 (FALSE C++11 ABI), Kornia 0.7.2 and timm 0.4.12. The environment dependency check passed. Microsoft Python, Python Environments and Jupyter extensions are installed in the WSL editor server.

The actual notebook preflight passed on the **NVIDIA GeForce GTX 1650 Ti (4 GB)**: strict pretrained VMamba loading, real 224 x 224 image input, adapter gradients, an actual backbone parameter update and predictions from all four baselines. A small real-image CS-MON staged training smoke test also passed with CUDA mixed precision, batch size 2, two data-loader workers and gradient accumulation. Its measured peak allocated GPU memory was approximately **0.78 GB**; that is a small smoke-test measurement, not a guarantee for every full experiment. A separate registered Jupyter kernel was launched successfully and verified CUDA availability and the GPU name.

**Use the VS Code window connected to WSL: Ubuntu-22.04**, open this project, and select **CS-MON GPU (WSL)** in the notebook kernel picker if it is not selected automatically. The notebook metadata and WSL editor interpreter setting point to this environment. Quick mode is configured; **Run All** now starts the actual five-model quick experiment. Full research training has not been executed during setup, and no final research checkpoint or performance claims have been produced by the smoke tests. The existing split manifest is ready.

Pretrained weights and package caches remain outside the project. The dataset and split assignments are unchanged. Label provenance and physical sample/batch/session independence remain unverified, as documented in notebook 01.


In [29]:
# Single editable configuration cell. Set label provenance only from actual records.
CONFIG = {
 'seed':42, 'mode':'quick', 'seeds':[42], 'full_seeds':[42,123,2026],
 'image_size':224, 'batch_size':None, 'num_workers':2, 'accumulation_steps':2,
 'quick_head_epochs':2, 'quick_finetune_epochs':3,
 'full_head_epochs':3, 'full_finetune_epochs':15, 'full_unfreeze_epochs':5,
 'head_lr':1e-3, 'backbone_lr':1e-5, 'weight_decay':1e-4,
 'use_pretrained':True, 'use_amp':True, 'gradient_clip':1.0,
 'use_early_stopping':True, 'early_stopping_patience':4,
 'run_baselines':True, 'run_ablations':False, 'full_run_ablations':True,
 'run_explainability':True, 'run_robustness':False, 'full_run_robustness':True,
 'run_latency_benchmark':True, 'run_embeddings':True, 'save_best_model':True,
 'augmentation':True, 'ig_steps':32, 'latency_warmup':10, 'latency_repeats':30,
 'allow_slow_reference_scan':False, 'vmamba_commit':'2ed52ead062a51a64521ed3871d52914bf532876',
 'vmamba_source_sha256':'572207a2b6553ff85d81b602eebda2041623073f01fc0c6d1280c5a2add040f6', 'checkpoint_sha256':'d855b57224a652bc692804932f88dc3a39b9028e64be99f745a7fd6792ca672a',
 'label_reference_documentation':'', 'allow_exploratory_labels':True,
 'allow_exploratory_split':True,
 'cache_dir':None,
}

## 1 : Environment, reproducibility and experiment contract
Seeds control ordering, initialization and spatial augmentation. Deterministic settings are enabled; CUDA extensions may still be nondeterministic, so results should be repeated. Batch size defaults conservatively to 2 below 6 GB GPU RAM, with accumulation for an effective batch of 4. Every primary model receives the same schedule and learning rates; these may be changed in configuration using validation data before final evaluation. Baseline partial fine-tuning uses its final feature block(s); full mode also unfreezes all features.

In [30]:
from pathlib import Path
import os, sys, copy, time, random, math, gc, json, hashlib, platform, urllib.request, importlib.util, ast
from contextlib import nullcontext
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import seaborn as sns
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import kornia.color as colour
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score,
                             mean_absolute_error, cohen_kappa_score, confusion_matrix, classification_report)
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
from IPython.display import display

ROOT=Path.cwd().resolve(); LABELS={'Low':0,'Medium':1,'High':2}; SWEETS=['Kesari','Muscat','Boondhi']
assert CONFIG['mode'] in ('quick','full') and CONFIG['use_pretrained'], 'Pretrained transfer learning is mandatory.'
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU_MEMORY=torch.cuda.get_device_properties(0).total_memory/2**30 if DEVICE.type=='cuda' else 0
BATCH_SIZE=CONFIG['batch_size'] or (2 if GPU_MEMORY<6 else 8 if GPU_MEMORY<12 else 16)
WORKERS=0 if os.name=='nt' else CONFIG['num_workers']
AMP=bool(CONFIG['use_amp'] and DEVICE.type=='cuda')
CACHE=Path(CONFIG['cache_dir'] or (Path.home()/'.cache'/'csmon'))
assert not CACHE.resolve().is_relative_to(ROOT), 'Keep dependency caches outside the project.'
CACHE.mkdir(parents=True,exist_ok=True)
torch.hub.set_dir(str(CACHE/'torch'))
sns.set_theme(style='whitegrid',palette='colorblind'); plt.rcParams.update({'figure.dpi':125,'figure.constrained_layout.use':True})
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True
    torch.use_deterministic_algorithms(True,warn_only=True)
seed_everything(CONFIG['seed'])
VERSIONS={'python':sys.version.split()[0],'torch':str(torch.__version__),
 'torchvision':__import__('torchvision').__version__, 'cuda':str(torch.version.cuda),
 'kornia':__import__('kornia').__version__, 'platform':platform.platform()}
HARDWARE={'device':str(DEVICE),'gpu':torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else platform.processor(),
 'gpu_memory_gb':GPU_MEMORY,'batch_size':BATCH_SIZE,'amp':AMP,'workers':WORKERS}
display(pd.Series(VERSIONS)); display(pd.Series(HARDWARE))
print('Quick results are preliminary.' if CONFIG['mode']=='quick' else 'Full experiment; test is reserved until all runs are trained.')

python                                                   3.10.12
torch                                                2.2.2+cu121
torchvision                                         0.17.2+cu121
cuda                                                        12.1
kornia                                                     0.8.2
platform       Linux-6.18.54.1-microsoft-standard-WSL2-x86_64...
dtype: object

device                                 cuda
gpu              NVIDIA GeForce GTX 1650 Ti
gpu_memory_gb                      3.999695
batch_size                                2
amp                                    True
workers                                   2
dtype: object

Quick results are preliminary.


## 2 : Manifest integrity and dynamic, colour-preserving transforms
Revalidate current file hashes against the preprocessing manifest to catch changed images. Recheck metadata and duplicate component separation. Spatial flips are the only training augmentation; colour conversions occur after the same spatial transform. Baselines receive that same RGB image. No processed-image cache is written. Validation/test have deterministic transforms. ImageNet normalization happens inside each model once.

In [1]:
manifest_path=ROOT/'outputs'/'split_manifest.csv'
assert manifest_path.is_file(), 'Execute notebook 01 first.'
manifest=pd.read_csv(manifest_path).fillna('')
assert {'relative_path','label','sweet','split','audit_component','sha256','pixel_sha256','label_provenance'}<=set(manifest)
assert manifest.relative_path.is_unique and manifest.label.isin([0,1,2]).all()
assert (manifest.colouring_intensity.map(LABELS)==manifest.label).all()
assert set(manifest.split)=={'train','validation','test'}
for col in ['audit_component','sha256','pixel_sha256','sample_id','preparation_batch_id','capture_session_id']:
    if col in manifest:
        valid=manifest[col].astype(str).str.strip().ne('')
        assert manifest.loc[valid].groupby(col).split.nunique().max()==1, f'Leakage: {col}'
for row in tqdm(manifest.itertuples(),total=len(manifest),desc='Verify manifest hashes'):
    p=(ROOT/row.relative_path).resolve()
    assert p.is_relative_to(ROOT) and p.is_file(), f'Invalid image path: {p}'
    assert hashlib.sha256(p.read_bytes()).hexdigest()==row.sha256, f'Data changed: {p}'
for split in ['train','validation','test']:
    part=manifest[manifest.split==split]
    assert set(part.label)=={0,1,2}
    assert len(part.groupby(['sweet','label']))==9, f'Insufficient {split} coverage'
if manifest.split_scope.str.contains('exploratory').any():
    assert CONFIG['allow_exploratory_split'], 'Supply reliable dependence metadata.'
    print('EXPLORATORY split: sample-independent generalization is not established.')
if not CONFIG['label_reference_documentation']:
    assert CONFIG['allow_exploratory_labels'], 'Supply controlled preparation/reference label documentation.'
    print('EXPLORATORY labels: CSV intensity provenance has not been independently verified.')
MANIFEST_SHA256=hashlib.sha256(manifest_path.read_bytes()).hexdigest()

class SweetsDataset(Dataset):
    def __init__(self,frame,augment=False): self.frame=frame.reset_index(drop=True); self.augment=augment
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        row=self.frame.iloc[i]
        with Image.open(ROOT/row.relative_path) as im:
            im=ImageOps.exif_transpose(im).convert('RGB').resize((CONFIG['image_size'],)*2,Image.Resampling.LANCZOS)
            if self.augment:
                if torch.rand(())<.5: im=ImageOps.mirror(im)
                if torch.rand(())<.5: im=ImageOps.flip(im)
            x=transforms.functional.to_tensor(im)
        return x,torch.tensor(int(row.label),dtype=torch.long),i

def make_loaders(seed,augment=True):
    generator=torch.Generator().manual_seed(seed)
    return {s:DataLoader(SweetsDataset(manifest[manifest.split==s],augment and s=='train'),
       batch_size=BATCH_SIZE,shuffle=s=='train',num_workers=WORKERS,pin_memory=DEVICE.type=='cuda',
       persistent_workers=WORKERS>0,generator=generator if s=='train' else None)
       for s in ['train','validation','test']}

class RGBNormalization(nn.Module):
    def __init__(self):
        super().__init__(); self.register_buffer('mean',torch.tensor([.485,.456,.406]).view(1,3,1,1))
        self.register_buffer('std',torch.tensor([.229,.224,.225]).view(1,3,1,1))
    def forward(self,x): return (x-self.mean)/self.std

NameError: name 'ROOT' is not defined

## 3 : Authentic VMamba initialization and strict checkpoint loading
The downloaded self-contained upstream file stays outside the project. Resolve `main` to a commit once, record its source hash and checkpoint hash, and reuse that commit for all models. For an exact rerun, set the printed commit and hashes in configuration. No upstream files are modified. For explicitly enabled reference diagnostics without Triton, the loader omits only unused top-level Triton JIT definitions in memory to bypass an upstream import-time decorator issue; the official VSSM, SS2D and PyTorch scan code remain unchanged. A checkpoint mismatch fails loudly. Retaining the complete upstream classifier norm and pool makes feature extraction explicit and avoids guessing a `forward_features` API.

Fast training requires `selective_scan_cuda` and compatible cross-scan support. The upstream reference scan is the authentic operation, but it is allowed only by the explicit slow-reference option. Use the upstream installation instructions above to resolve missing extensions. Downloads execute official upstream Python code and load a trusted official checkpoint; verify configured hashes for locked experiments.

In [32]:
CHECKPOINT_URL='https://github.com/MzeroMiko/VMamba/releases/download/%23v2cls/vssm1_tiny_0230s_ckpt_epoch_264.pth'
VMAMBA_KWARGS=dict(depths=[2,2,8,2],dims=96,ssm_d_state=1,ssm_ratio=1.0,ssm_dt_rank='auto',
 ssm_conv=3,ssm_conv_bias=False,forward_type='v05_noz',mlp_ratio=4.0,
 downsample_version='v3',patchembed_version='v2',norm_layer='ln2d',drop_path_rate=.2,
 use_checkpoint=False,num_classes=1000)
VMAMBA_MODULE=None; VMAMBA_PROVENANCE={}

def download(url,destination,expected=None):
    if not destination.is_file():
        request=urllib.request.Request(url,headers={'User-Agent':'CSMON-research'})
        try:
            with urllib.request.urlopen(request,timeout=120) as source, destination.open('wb') as target:
                while chunk:=source.read(1024*1024): target.write(chunk)
        except Exception:
            destination.unlink(missing_ok=True); raise
    digest=hashlib.sha256(destination.read_bytes()).hexdigest()
    assert expected is None or digest==expected, f'Integrity mismatch: {destination}'
    return digest

def require_triton_ptxas():
    try:
        from triton.common.backend import path_to_ptxas
        path_to_ptxas()
    except RuntimeError as exc:
        if 'Cannot find ptxas' not in str(exc):
            raise
        raise RuntimeError(
            'Triton is available, but its CUDA assembler (ptxas) is missing. '
            'Install the CUDA toolkit matching this PyTorch/CUDA environment, '
            'or set TRITON_PTXAS_PATH to the full path of ptxas; then restart '
            'the kernel and rerun the preflight.'
        ) from exc

def initialize_vmamba(require_checkpoint=True):
    global VMAMBA_MODULE,VMAMBA_PROVENANCE
    if VMAMBA_MODULE is not None:
        if VMAMBA_PROVENANCE.get('fast_scan'):
            require_triton_ptxas()
        if require_checkpoint:
            digest=download(CHECKPOINT_URL,CACHE/'vssm1_tiny_0230s_ckpt_epoch_264.pth',CONFIG['checkpoint_sha256'])
            VMAMBA_PROVENANCE['checkpoint_sha256']=digest
        return
    commit=CONFIG['vmamba_commit']
    if commit=='main':
        req=urllib.request.Request('https://api.github.com/repos/MzeroMiko/VMamba/commits/main',headers={'User-Agent':'CSMON-research'})
        with urllib.request.urlopen(req,timeout=60) as response: commit=json.load(response)['sha']
    assert len(commit)==40 and all(c in '0123456789abcdef' for c in commit), 'Use an official commit SHA.'
    source=CACHE/f'vmamba-{commit}.py'
    source_hash=download(f'https://raw.githubusercontent.com/MzeroMiko/VMamba/{commit}/vmamba.py',source,CONFIG['vmamba_source_sha256'])
    spec=importlib.util.spec_from_file_location('csmon_official_vmamba',source)
    module=importlib.util.module_from_spec(spec)
    sys.modules[spec.name]=module
    if importlib.util.find_spec('triton') is None:
        if not CONFIG['allow_slow_reference_scan']:
            raise RuntimeError('Triton is unavailable. Use WSL2/Linux with the matching CUDA dependencies; see setup notes. Enable allow_slow_reference_scan explicitly for reference diagnostics.')
        # Upstream references @triton.jit unconditionally despite its fallback flag.
        # Omit only unused Triton JIT definitions in memory; all VSSM/SS2D and torch scans are unchanged.
        tree=ast.parse(source.read_text(encoding='utf-8'))
        tree.body=[node for node in tree.body if not (isinstance(node,(ast.FunctionDef,ast.AsyncFunctionDef)) and
                   any(ast.unparse(d).startswith('triton.') for d in node.decorator_list))]
        exec(compile(tree,str(source),'exec'),module.__dict__)
    else:
        spec.loader.exec_module(module)
    fast_scan=bool(getattr(module,'WITH_SELECTIVESCAN_MAMBA',False)) and DEVICE.type=='cuda'
    if not fast_scan and not CONFIG['allow_slow_reference_scan']:
        raise RuntimeError('Fast selective_scan_cuda is unavailable. Use WSL2/Linux + matching PyTorch/CUDA/mamba-ssm; see setup notes. Explicitly enable allow_slow_reference_scan only for diagnostics.')
    if fast_scan:
        require_triton_ptxas()
    if not fast_scan: print('Explicit diagnostic mode: official PyTorch reference selective scan, potentially very slow.')
    checkpoint=CACHE/'vssm1_tiny_0230s_ckpt_epoch_264.pth'
    checkpoint_hash=download(CHECKPOINT_URL,checkpoint,CONFIG['checkpoint_sha256']) if require_checkpoint else None
    VMAMBA_PROVENANCE={'commit':commit,'source_sha256':source_hash,'checkpoint_url':CHECKPOINT_URL,
                       'checkpoint_sha256':checkpoint_hash,'fast_scan':fast_scan}
    VMAMBA_MODULE=module
    display(pd.Series(VMAMBA_PROVENANCE))

def build_vmamba(pretrained=True):
    initialize_vmamba(require_checkpoint=pretrained)
    net=VMAMBA_MODULE.VSSM(**VMAMBA_KWARGS)
    if pretrained:
        # Official release stores its ImageNet model under 'model'. No relaxed key matching.
        payload=torch.load(CACHE/'vssm1_tiny_0230s_ckpt_epoch_264.pth',map_location='cpu',weights_only=False)
        assert 'model' in payload, 'Unexpected official checkpoint format.'
        net.load_state_dict(payload['model'],strict=True)
        del payload
    assert net.num_features==768
    net.classifier.head=nn.Identity()
    return net

## 4 : Sample-dependent RGB/HSV/LAB fusion and monotonic CORAL head
Kornia performs fixed differentiable conversions in float32: HSV hue radians are divided by 2*pi; LAB uses D65 L*,a*,b*. Each representation has a learned 1 x 1 projection. A pooled-content gate produces three softmax weights per image. A zero-initialized 3-channel output adapter adds a bounded residual to raw RGB, preserving the initial pretrained input exactly. RGB is normalized once after fusion. Gate weights are architectural mixture weights, not causal importance scores.

The shared scalar score has ordered cutpoints `t0` and `t0 + softplus(gap)`. Thus logits and cumulative probabilities are ordered for every input. The loss is binary cross entropy averaged over the two cumulative targets. Class probabilities are derived by differences; prediction uses the maximum class probability consistently for all models. Ordinal MAE uses these predicted indices.

In [ ]:
def colour_representations(x):
    x=x.float()
    hsv=colour.rgb_to_hsv(x); hsv=torch.cat([hsv[:,:1]/(2*math.pi),hsv[:,1:]],dim=1)
    lab=colour.rgb_to_lab(x)
    lab=torch.cat([lab[:,:1]/50-1,lab[:,1:]/128],dim=1)
    return x*2-1,hsv*2-1,lab

class ColourFusion(nn.Module):
    def __init__(self,channels=12,gated=True):
        super().__init__(); self.gated=gated
        self.projections=nn.ModuleList([nn.Sequential(nn.Conv2d(3,channels,1),nn.GELU()) for _ in range(3)])
        self.gate=nn.Sequential(nn.Linear(3*channels,channels),nn.GELU(),nn.Linear(channels,3))
        self.adapter=nn.Conv2d(channels,3,1)
        nn.init.zeros_(self.adapter.weight); nn.init.zeros_(self.adapter.bias)
        self.last_gates=None
    def forward(self,x):
        assert x.ndim==4 and x.shape[1]==3
        with torch.autocast(device_type=x.device.type,enabled=False): reps=colour_representations(x)
        features=[projection(rep) for projection,rep in zip(self.projections,reps)]
        context=torch.cat([f.mean((2,3)) for f in features],dim=1)
        gates=self.gate(context).softmax(dim=1) if self.gated else context.new_full((x.shape[0],3),1/3)
        fused=sum(f*gates[:,i,None,None,None] for i,f in enumerate(features))
        self.last_gates=gates.detach()
        adapted=x+.1*torch.tanh(self.adapter(fused))
        assert adapted.shape==x.shape
        return adapted

class OrdinalHead(nn.Module):
    def __init__(self,features):
        super().__init__(); self.score=nn.Linear(features,1,bias=False)
        # Start with thresholds [-1, 1]. A gap <= log(4) makes Medium
        # unable to be the uniquely most probable class under argmax.
        # Both thresholds remain learnable; no predictions are forced.
        initial_gap=2.0
        self.first=nn.Parameter(torch.tensor(-initial_gap/2))
        self.raw_gap=nn.Parameter(torch.tensor(math.log(math.expm1(initial_gap-1e-4))))
    def forward(self,x):
        thresholds=torch.stack([self.first,self.first+F.softplus(self.raw_gap)+1e-4])
        return self.score(x)-thresholds

def ordinal_targets(y): return (y[:,None]>torch.arange(2,device=y.device)[None,:]).float()
def ordinal_loss(logits,y): return F.binary_cross_entropy_with_logits(logits,ordinal_targets(y))
def class_probabilities(logits,ordinal=True):
    if not ordinal: return logits.float().softmax(1)
    cumulative=logits.float().sigmoid()
    return torch.stack([1-cumulative[:,0],cumulative[:,0]-cumulative[:,1],cumulative[:,1]],dim=1)

class ResearchModel(nn.Module):
    def __init__(self,backbone,features,fusion=None,ordinal=True,is_vmamba=False):
        super().__init__(); self.backbone=backbone; self.fusion=fusion
        self.normalization=RGBNormalization(); self.ordinal=ordinal; self.is_vmamba=is_vmamba
        self.head=OrdinalHead(features) if ordinal else nn.Linear(features,3)
        self.frozen_backbone=False; self.frozen_modules=[]
    def features(self,x):
        if self.fusion is not None: x=self.fusion(x)
        return self.backbone(self.normalization(x))
    def forward(self,x): return self.head(self.features(x))
    def train(self,mode=True):
        super().train(mode)
        if mode:
            if self.frozen_backbone: self.backbone.eval()
            for module in self.frozen_modules: module.eval()
        return self

BASELINE_NAMES=['MobileNetV3-Small','ResNet18','ShuffleNetV2 x1.0','MobileNetV2']
CSMON='CS-MON (Fine-Tuned VMamba-Tiny)'
def build_baseline(name):
    if name=='MobileNetV3-Small':
        b=models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        d=b.classifier[0].in_features; b.classifier=nn.Identity()
    elif name=='ResNet18':
        b=models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1); d=b.fc.in_features; b.fc=nn.Identity()
    elif name=='ShuffleNetV2 x1.0':
        b=models.shufflenet_v2_x1_0(weights=models.ShuffleNet_V2_X1_0_Weights.IMAGENET1K_V1); d=b.fc.in_features; b.fc=nn.Identity()
    elif name=='MobileNetV2':
        b=models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1); d=b.classifier[1].in_features; b.classifier=nn.Identity()
    else: raise ValueError(name)
    return ResearchModel(b,d)

def build_csmon(fusion='gated',ordinal=True,pretrained=True):
    f=None if fusion=='rgb' else ColourFusion(gated=fusion=='gated')
    return ResearchModel(build_vmamba(pretrained),768,f,ordinal,is_vmamba=True)

# Algebraic unit-style checks; no synthetic observations are used for training.
assert ordinal_targets(torch.tensor([0,1,2])).tolist()==[[0.,0.],[1.,0.],[1.,1.]]
h=OrdinalHead(5); logits=h(torch.randn(7,5)); probs=class_probabilities(logits)
assert logits.shape==(7,2) and torch.all(logits[:,0]>=logits[:,1])
assert torch.all(probs>=0) and torch.allclose(probs.sum(1),torch.ones(7),atol=1e-6)
assert torch.isfinite(ordinal_loss(logits,torch.tensor([0,1,2,0,1,2,0])))
# Regression check: every class can win with the initial ordinal thresholds.
# These scalar scores check the head algebra only, not model performance.
with torch.no_grad():
    probe_head=OrdinalHead(1)
    probe_head.score.weight.fill_(1.)
    probe_probs=class_probabilities(probe_head(torch.tensor([[-5.], [0.], [5.]])))
    assert probe_probs.argmax(1).tolist()==[0,1,2], 'Initial ordinal head excludes a class.'
    assert torch.allclose(probe_probs.sum(1),torch.ones(3),atol=1e-6)
del probe_head,probe_probs
# Check colour ranges against known red/white/black and skimage reference on real pixels later.
known=torch.tensor([[[[1.,1.,0.]],[[0.,1.,0.]],[[0.,1.,0.]]]])
reps=colour_representations(known)
assert torch.allclose(reps[1][0,:,0,0],torch.tensor([-1.,1.,1.]),atol=1e-4)
assert torch.allclose(reps[2][0,0,0,1],torch.tensor(1.),atol=1e-3)

## 5 : Staged optimization, validation selection and metrics
Stage 1 freezes feature weights and running statistics but preserves input gradients through the backbone so fusion can learn. Stage 2 trains final VMamba layers (stages 3-4) or each baseline's final feature blocks. Stage 3 in full mode unfreezes all features at half the backbone learning rate. Optimizers exclude frozen parameters and omit decay on biases, normalization/scalar parameters and upstream parameters marked `_no_weight_decay`.

Only stage 2/3 checkpoints compete for the final selected model. This prevents a frozen-backbone epoch from being saved as a purported fine-tuned model. The first optimized backbone tensor must actually change. Validation macro F1 selects checkpoints, with validation loss breaking ties. Early stopping resets by stage. Cosine scheduling resets per stage. Test inference occurs only after all planned training, seeds and ablations finish. Accumulation correctly weights the final partial window.

Ordinal heads start with learnable thresholds [-1, 1] (gap 2), allowing Low, Medium and High each to be the most probable class. Training reports the learned gap, validation Medium prediction count and Medium recall each epoch. A gap at or below log(4) is flagged because it excludes a unique Medium argmax. This initialization applies to all primary models; loading an older checkpoint restores its old learned thresholds. Retrain from pretrained ImageNet weights to apply the change. The ordinal loss, probability conversion and checkpoint selection criterion are unchanged.


In [ ]:
def set_stage(model,stage):
    model.frozen_backbone=stage=='head'; model.frozen_modules=[]
    for p in model.backbone.parameters(): p.requires_grad_(stage=='full')
    if stage=='partial':
        b=model.backbone
        if model.is_vmamba: blocks=list(b.layers)[-2:]; model.frozen_modules=[b.patch_embed,*list(b.layers)[:-2]]
        elif hasattr(b,'layer4'): blocks=[b.layer4]; model.frozen_modules=[b.conv1,b.bn1,b.layer1,b.layer2,b.layer3]
        elif hasattr(b,'stage4'): blocks=[b.stage4,b.conv5]; model.frozen_modules=[b.conv1,b.stage2,b.stage3]
        else: blocks=list(b.features)[-3:]; model.frozen_modules=list(b.features)[:-3]
        for block in blocks:
            for p in block.parameters(): p.requires_grad_(True)
        if model.is_vmamba:
            for p in b.classifier.parameters(): p.requires_grad_(True)
    for p in model.head.parameters(): p.requires_grad_(True)
    if model.fusion is not None:
        for p in model.fusion.parameters(): p.requires_grad_(True)
    return model

def optimizer_for(model,stage):
    groups={}
    for name,p in model.named_parameters():
        if not p.requires_grad: continue
        backbone=name.startswith('backbone.')
        lr=(CONFIG['backbone_lr']*(.5 if stage=='full' else 1)) if backbone else CONFIG['head_lr']
        decay=0. if p.ndim<2 or name.endswith('.bias') or getattr(p,'_no_weight_decay',False) else CONFIG['weight_decay']
        key=(backbone,lr,decay)
        groups.setdefault(key,[]).append(p)
    return torch.optim.AdamW([{'params':ps,'lr':lr,'weight_decay':decay,'role':'backbone' if bb else 'new'}
                              for (bb,lr,decay),ps in groups.items()])

def calibration(y,probs,bins=10):
    confidence=probs.max(1); correct=probs.argmax(1)==y; rows=[]; ece=0.
    for i in range(bins):
        mask=(confidence>=i/bins)&((confidence<(i+1)/bins) if i<bins-1 else (confidence<=1))
        n=int(mask.sum())
        if n:
            acc=float(correct[mask].mean()); conf=float(confidence[mask].mean())
            ece+=n/len(y)*abs(acc-conf); rows.append((conf,acc,n))
    return float(ece),pd.DataFrame(rows,columns=['confidence','accuracy','count'])

def metrics(y,probs):
    pred=probs.argmax(1)
    p,r,f,_=precision_recall_fscore_support(y,pred,labels=[0,1,2],average='macro',zero_division=0)
    qwk=cohen_kappa_score(y,pred,labels=[0,1,2],weights='quadratic') if len(np.unique(np.r_[y,pred]))>1 else float('nan')
    return {'Accuracy':accuracy_score(y,pred),'Macro precision':p,'Macro recall':r,'Macro F1':f,
            'Weighted F1':f1_score(y,pred,labels=[0,1,2],average='weighted',zero_division=0),
            'Ordinal MAE':mean_absolute_error(y,pred),'QWK':qwk,'ECE':calibration(y,probs)[0]}

def loss_for(model,logits,y): return ordinal_loss(logits,y) if model.ordinal else F.cross_entropy(logits,y)
def autocast(): return torch.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=AMP)
def cpu_state(model): return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

@torch.no_grad()
def predict(model,loader,collect_features=False):
    model.eval(); ys=[]; ps=[]; indices=[]; gates=[]; features=[]; total_loss=0
    for x,y,idx in loader:
        x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
        with autocast():
            f=model.features(x); logits=model.head(f); loss=loss_for(model,logits,y)
        prob=class_probabilities(logits,model.ordinal)
        assert prob.shape==(len(y),3) and torch.isfinite(prob).all()
        assert (prob>=-1e-6).all() and torch.allclose(prob.sum(1),torch.ones(len(y),device=DEVICE),atol=1e-5)
        ys.append(y.cpu().numpy()); ps.append(prob.cpu().numpy()); indices.append(idx.numpy())
        total_loss+=loss.item()*len(y)
        if collect_features: features.append(f.float().cpu().numpy())
        if model.fusion is not None: gates.append(model.fusion.last_gates.float().cpu().numpy())
    result={'y':np.concatenate(ys),'probs':np.concatenate(ps),'indices':np.concatenate(indices),'loss':total_loss/len(loader.dataset)}
    if gates: result['gates']=np.concatenate(gates)
    if features: result['features']=np.concatenate(features)
    result['metrics']=metrics(result['y'],result['probs'])
    return result

def train_model(model,loaders):
    model.to(DEVICE)
    scaler=torch.cuda.amp.GradScaler(enabled=AMP)
    prefix=CONFIG['mode']; schedule=[('head',CONFIG[prefix+'_head_epochs']),('partial',CONFIG[prefix+'_finetune_epochs'])]
    if prefix=='full' and CONFIG['full_unfreeze_epochs']>0: schedule.append(('full',CONFIG['full_unfreeze_epochs']))
    assert schedule[1][1]>0
    history=[]; best_state=None; best_key=(-float('inf'),-float('inf')); selected={}; epoch_number=0
    start=time.perf_counter(); backbone_updated=False
    for stage,epochs in schedule:
        if epochs<=0: continue
        set_stage(model,stage); optimizer=optimizer_for(model,stage)
        scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=epochs)
        stage_best=(-float('inf'),-float('inf')); stale=0
        for _ in range(epochs):
            epoch_number+=1; model.train(); optimizer.zero_grad(set_to_none=True); total=0.; seen=0
            lr_roles={role:max([g['lr'] for g in optimizer.param_groups if g['role']==role],default=0.) for role in ['backbone','new']}
            bar=tqdm(loaders['train'],desc=f'{stage} : epoch {epoch_number}',leave=False)
            for batch,(x,y,_) in enumerate(bar):
                x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
                accumulation=CONFIG['accumulation_steps']
                window_start=(batch//accumulation)*accumulation
                # Denominator counts samples in this accumulation window, including the last small batch.
                first_sample=window_start*BATCH_SIZE
                window_samples=min(accumulation*BATCH_SIZE,len(loaders['train'].dataset)-first_sample)
                with autocast(): logits=model(x); loss=loss_for(model,logits,y)
                assert torch.isfinite(loss), 'Nonfinite training loss.'
                scaler.scale(loss*len(y)/window_samples).backward()
                if (batch+1)%accumulation==0 or batch+1==len(loaders['train']):
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],CONFIG['gradient_clip'])
                    probe=None
                    if stage!='head' and not backbone_updated:
                        probe=next(p for p in model.backbone.parameters() if p.requires_grad and p.grad is not None and p.grad.abs().sum()>0)
                        before=probe.detach().clone()
                    scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
                    if probe is not None: backbone_updated=not torch.equal(before,probe.detach())
                total+=float(loss.detach())*len(y); seen+=len(y); bar.set_postfix(loss=total/seen)
            validation=predict(model,loaders['validation']); score=validation['metrics']['Macro F1']
            validation_pred=validation['probs'].argmax(1)
            medium_mask=validation['y']==LABELS['Medium']
            medium_count=int((validation_pred==LABELS['Medium']).sum())
            medium_recall=float((validation_pred[medium_mask]==LABELS['Medium']).mean()) if medium_mask.any() else float('nan')
            ordinal_gap=float(F.softplus(model.head.raw_gap.detach()).cpu())+1e-4 if model.ordinal else float('nan')
            if model.ordinal and ordinal_gap<=math.log(4):
                warnings_message='Ordinal threshold gap <= log(4): Medium cannot be uniquely most probable.'
                print('WARNING:',warnings_message)
            key=(score,-validation['loss'])
            history.append(dict(epoch=epoch_number,stage=stage,train_loss=total/seen,val_loss=validation['loss'],
                                val_f1=score,backbone_lr=lr_roles['backbone'],new_lr=lr_roles['new'],
                                ordinal_gap=ordinal_gap,val_medium_predictions=medium_count,val_medium_recall=medium_recall))
            if stage!='head' and backbone_updated and key>best_key:
                best_key=key; best_state=cpu_state(model); selected={'epoch':epoch_number,'stage':stage,
                    'validation_macro_f1':score,'validation_loss':validation['loss'],
                    'ordinal_gap':ordinal_gap,'validation_medium_predictions':medium_count,'validation_medium_recall':medium_recall}
            if key>stage_best: stage_best=key; stale=0
            else: stale+=1
            scheduler.step()
            print(f"Epoch {epoch_number:02d} {stage}: train {total/seen:.4f}, val {validation['loss']:.4f}, macro F1 {score:.4f}; gap {ordinal_gap:.4f}, validation Medium predictions {medium_count}, Medium recall {medium_recall:.4f}")
            if CONFIG['use_early_stopping'] and stale>=CONFIG['early_stopping_patience']: break
    assert backbone_updated and best_state is not None, 'No verified backbone update; cannot save a frozen feature extractor as fine-tuned.'
    model.load_state_dict(best_state,strict=True)
    return {'state':best_state,'history':pd.DataFrame(history),'selected':selected,
            'training_seconds':time.perf_counter()-start,'parameters':sum(p.numel() for p in model.parameters()),
            'backbone_updated':backbone_updated}

## 6 : Real-image preflight before expensive experiments
The preflight checks colour conversions against scikit-image, exact initial fusion identity, pretrained loading, feature shapes, finite gradients into the adapter through a frozen backbone, one actual fine-tuning update, all four baseline outputs, and a strict in-memory state roundtrip. The preflight model is discarded, and fresh seeded models are built for the experiment. Nothing is measured on test images. This cell is expected to stop with actionable guidance when the required VMamba backend is absent.

In [ ]:
def preflight():
    import triton.common.backend as triton_backend
    try:
        triton_backend.path_to_ptxas()
    except RuntimeError as exc:
        if 'Cannot find ptxas' not in str(exc):
            raise
        for entry in sys.path:
            candidate=Path(entry)/'nvidia'/'cuda_nvcc'/'bin'/'ptxas'
            if candidate.is_file():
                ptxas_path=candidate
                if ' ' in str(candidate):
                    alias=CACHE/f"ptxas-{hashlib.sha256(str(candidate.resolve()).encode()).hexdigest()[:12]}"
                    if not alias.exists(): alias.symlink_to(candidate.resolve())
                    if alias.resolve()!=candidate.resolve():
                        raise RuntimeError(f'Cannot safely expose ptxas to Triton: {alias}')
                    ptxas_path=alias
                os.environ['TRITON_PTXAS_PATH']=str(ptxas_path)
                triton_backend.path_to_ptxas.cache_clear()
                break
        try:
            triton_backend.path_to_ptxas()
        except RuntimeError as exc:
            if 'Cannot find ptxas' not in str(exc):
                raise
            raise RuntimeError(
                'Triton cannot find ptxas. Install nvidia-cuda-nvcc in this kernel '
                'environment or configure TRITON_PTXAS_PATH, then rerun the preflight.'
            ) from exc
    from skimage.color import rgb2hsv,rgb2lab
    loaders=make_loaders(CONFIG['seed'],augment=False)
    x,y,_=next(iter(loaders['train'])); x=x[:1].to(DEVICE); y=y[:1].to(DEVICE)
    a,b,c=colour_representations(x)
    image=x[0].permute(1,2,0).cpu().numpy()
    np.testing.assert_allclose(((b[0].permute(1,2,0).cpu().numpy()+1)/2),rgb2hsv(image),atol=2e-4)
    lab=rgb2lab(image); scaled=np.concatenate([lab[...,:1]/50-1,lab[...,1:]/128],axis=-1)
    np.testing.assert_allclose(c[0].permute(1,2,0).cpu().numpy(),scaled,atol=2e-3)
    model=build_csmon().to(DEVICE)
    print('Preflight: pretrained VMamba loaded; checking colour fusion and gradients.',flush=True)
    assert torch.equal(model.fusion(x),x), 'Initial fusion must preserve RGB exactly.'
    set_stage(model,'head'); model.train(); model.zero_grad(set_to_none=True)
    logits=model(x); loss=ordinal_loss(logits,y); loss.backward()
    assert logits.shape==(1,2) and torch.isfinite(loss)
    assert model.fusion.adapter.weight.grad is not None and model.fusion.adapter.weight.grad.abs().sum()>0
    assert all(p.grad is None for p in model.backbone.parameters())
    print('Preflight: frozen-backbone fusion gradient passed.',flush=True)
    set_stage(model,'partial'); model.train(); model.zero_grad(set_to_none=True)
    optimizer=optimizer_for(model,'partial')
    logits=model(x); loss=ordinal_loss(logits,y); loss.backward()
    probe=next(p for p in model.backbone.parameters() if p.requires_grad and p.grad is not None and p.grad.abs().sum()>0)
    before=probe.detach().clone(); optimizer.step(); assert not torch.equal(before,probe.detach())
    print('Preflight: actual VMamba parameter update passed.',flush=True)
    model.eval()
    with torch.no_grad(): expected=model(x).detach().cpu()
    state=cpu_state(model); model.load_state_dict(state,strict=True)
    with torch.no_grad(): actual=model(x).detach().cpu()
    torch.testing.assert_close(expected,actual)
    del model,state,optimizer; gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
    for name in BASELINE_NAMES:
        model=build_baseline(name).to(DEVICE).eval()
        with torch.no_grad(): logits=model(x); prob=class_probabilities(logits)
        assert logits.shape==(1,2) and torch.isfinite(prob).all() and (prob>=0).all()
        torch.testing.assert_close(prob.sum(1),torch.ones(1,device=DEVICE))
        del model; gc.collect()
    print('Preflight passed on a real training image, including a verified VMamba parameter update.')
preflight()

## 7 : Predeclared primary models, optional ablations and multiple seeds
A0: RGB + softmax; A1: RGB + CORAL; A2: fixed equal mixture of learned colour projections + CORAL; A3: adaptive mixture + CORAL, without augmentation; A4: A3 with colour-preserving spatial flips. This makes A0 -> A1 isolate ordinal learning, A1 -> A2 multi-space fusion, A2 -> A3 gating and A3 -> A4 augmentation. A2 retains trainable projections, changing only its gate to fixed equal weights. The primary CS-MON uses augmentation by default, so it is exactly A4; its result is reused. If augmentation is disabled it is A3 and that result is reused instead.

Full mode enables selected ablations and metadata robustness by default via the separate full-mode flags. All seeds use the same held-out manifest split, so variation describes training randomness, not independent dataset replications. The best primary seed is selected on validation only for checkpoint saving and detailed plots. Aggregate test metrics across seeds are also reported. Do not use test comparisons to tune subsequent runs.

In [ ]:
def experiment_plan():
    plan={CSMON:dict(kind='vmamba',fusion='gated',ordinal=True,augment=CONFIG['augmentation'])}
    if CONFIG['run_baselines']:
        for name in BASELINE_NAMES: plan[name]=dict(kind='baseline',augment=CONFIG['augmentation'])
    ablations=CONFIG['mode']=='full' and CONFIG['full_run_ablations'] or CONFIG['mode']!='quick' and CONFIG['run_ablations']
    if ablations:
        for name,fusion,ordinal,augment in [('A0','rgb',False,False),('A1','rgb',True,False),
                                          ('A2','fixed',True,False),('A3','gated',True,False),('A4','gated',True,True)]:
            plan[name]=dict(kind='vmamba',fusion=fusion,ordinal=ordinal,augment=augment)
    return plan

def construct(spec,name,pretrained=True):
    if spec['kind']=='baseline': return build_baseline(name)
    return build_csmon(spec['fusion'],spec['ordinal'],pretrained)

PLAN=experiment_plan()
SEEDS=list(dict.fromkeys(CONFIG['full_seeds'] if CONFIG['mode']=='full' else CONFIG['seeds']))
assert SEEDS
RUNS={}; aliases={}; execution_cache={}
for seed in SEEDS:
    for name,spec in PLAN.items():
        signature=(seed,name if spec['kind']=='baseline' else 'vmamba',json.dumps(spec,sort_keys=True))
        if signature in execution_cache:
            aliases[(seed,name)]=execution_cache[signature]; continue
        seed_everything(seed); loaders=make_loaders(seed,spec['augment'])
        model=construct(spec,name)
        print(f'\nTraining {name} : seed {seed} : {CONFIG["mode"]}')
        run=train_model(model,loaders)
        run.update(name=name,seed=seed,spec=spec,versions=copy.deepcopy(VERSIONS))
        RUNS[(seed,name)]=run; execution_cache[signature]=(seed,name)
        del model,loaders; gc.collect()
        if DEVICE.type=='cuda': torch.cuda.empty_cache()

primary_keys=[key for key in RUNS if key[1]==CSMON]
SELECTED_KEY=max(primary_keys,key=lambda key:(RUNS[key]['selected']['validation_macro_f1'],-RUNS[key]['selected']['validation_loss']))
print('Primary checkpoint selected by validation only:',SELECTED_KEY,RUNS[SELECTED_KEY]['selected'])

## 8 : One final held-out evaluation per trained configuration
Latency is batch-one model inference including fusion/conversion/normalization, excluding file loading. It uses warm-up and CUDA synchronization, with measured hardware recorded. It is not a smartphone benchmark. Baseline checkpoints remain in CPU memory only until this evaluation; only CS-MON is serialized. Each trained configuration is evaluated once and aliases reuse its result. Reload construction uses the same architecture; test predictions never select epochs or seeds.

In [ ]:
def synchronize():
    if DEVICE.type=='cuda': torch.cuda.synchronize()

@torch.no_grad()
def latency_ms(model,loader):
    if not CONFIG['run_latency_benchmark']: return float('nan')
    x=next(iter(loader))[0][:1].to(DEVICE); model.eval()
    for _ in range(CONFIG['latency_warmup']):
        with autocast(): model(x)
    synchronize(); measurements=[]
    for _ in range(CONFIG['latency_repeats']):
        synchronize(); start=time.perf_counter()
        with autocast(): model(x)
        synchronize(); measurements.append((time.perf_counter()-start)*1000)
    return float(np.median(measurements))

RESULTS={}; test_loader=make_loaders(CONFIG['seed'],False)['test']
for key,run in RUNS.items():
    model=construct(run['spec'],run['name'],pretrained=False).to(DEVICE)
    model.load_state_dict(run['state'],strict=True)
    evaluation=predict(model,test_loader,collect_features=run['name']==CSMON)
    evaluation['metrics'].update({'Parameters':run['parameters'],'Training seconds':run['training_seconds'],
                                  'Latency ms (batch 1)':latency_ms(model,test_loader)})
    RESULTS[key]=evaluation
    del model; gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
for alias,target in aliases.items(): RESULTS[alias]=RESULTS[target]
rows=[dict(Model=name,Seed=seed,**result['metrics']) for (seed,name),result in RESULTS.items()]
all_results=pd.DataFrame(rows)
primary_names=BASELINE_NAMES+[CSMON] if CONFIG['run_baselines'] else [CSMON]
primary_results=all_results[all_results.Model.isin(primary_names)]
comparison=primary_results.groupby('Model',sort=False).mean(numeric_only=True).drop(columns='Seed').reindex(primary_names)
columns=['Accuracy','Macro F1','Ordinal MAE','QWK','Parameters','Latency ms (batch 1)']
styled=comparison[columns].style.format(precision=4).highlight_max(subset=['Accuracy','Macro F1','QWK'],color='#d4edda').highlight_min(subset=['Ordinal MAE','Parameters','Latency ms (batch 1)'],color='#d4edda')
display(styled)
if len(SEEDS)>1:
    display(primary_results.groupby('Model').agg({m:['mean','std'] for m in ['Accuracy','Macro F1','Ordinal MAE','QWK','ECE']}))
print('Latency hardware:',HARDWARE,'; batch=1, median of',CONFIG['latency_repeats'],'runs')
selected_run=RUNS[SELECTED_KEY]; selected_result=RESULTS[SELECTED_KEY]
selected_model=construct(selected_run['spec'],CSMON,pretrained=False).to(DEVICE)
selected_model.load_state_dict(selected_run['state'],strict=True); selected_model.eval()
# Release baseline states now that evaluation is complete. They are never written to disk.
for key,run in RUNS.items():
    if key!=SELECTED_KEY: run.pop('state',None)

## 9 : Training, stage transitions and five-model comparison
These plots use measured training histories and final test metrics. With repeated seeds, bars show the mean and one standard deviation. Missing/undefined metrics remain missing. Detailed confusion matrices use the preselected CS-MON seed consistently across models.

In [ ]:
history=selected_run['history']
fig,axes=plt.subplots(1,2,figsize=(13,4))
axes[0].plot(history.epoch,history.train_loss,label='Training'); axes[0].plot(history.epoch,history.val_loss,label='Validation')
axes[0].set(title='CS-MON loss and fine-tuning stages',xlabel='Epoch',ylabel='Cumulative BCE')
for stage,part in history.groupby('stage',sort=False):
    axes[0].axvline(part.epoch.iloc[0]-.5,color='gray',alpha=.5,linestyle=':')
    axes[0].text(part.epoch.mean(),.95,stage,transform=axes[0].get_xaxis_transform(),ha='center',va='top')
axes[0].legend()
axes[1].plot(history.epoch,history.backbone_lr,label='Backbone'); axes[1].plot(history.epoch,history.new_lr,label='New modules')
axes[1].set(title='Learning rates', xlabel='Epoch', ylabel='Learning rate')
axes[1].set_yscale('symlog', linthresh=1e-7)
axes[1].legend()
plt.show()
short_names={'MobileNetV3-Small':'MobileNetV3-S','ResNet18':'ResNet18','ShuffleNetV2 x1.0':'ShuffleNetV2','MobileNetV2':'MobileNetV2',CSMON:'CS-MON'}
def comparison_bars(metrics_to_plot,title):
    fig,axes=plt.subplots(1,len(metrics_to_plot),figsize=(4.1*len(metrics_to_plot),4.5),squeeze=False)
    for ax,metric in zip(axes[0],metrics_to_plot):
        means=comparison[metric]
        std=primary_results.groupby('Model')[metric].std().reindex(primary_names).fillna(0)
        ax.bar(range(len(means)),means.to_numpy(),yerr=std.to_numpy(),capsize=3,color=sns.color_palette('colorblind',len(means)))
        ax.set_xticks(range(len(means)),[short_names[n] for n in primary_names],rotation=30,ha='right')
        ax.set_title(metric+(' (lower is better)' if metric in ['Ordinal MAE','ECE'] else '')); ax.set_ylabel(metric)
    fig.suptitle(title); plt.show()
comparison_bars(['Accuracy','Macro F1','QWK','Ordinal MAE'],'Held-out performance : '+CONFIG['mode'])
comparison_bars(['Parameters','Training seconds','Latency ms (batch 1)'],'Measured model efficiency')
selected_seed=SELECTED_KEY[0]
fig,axes=plt.subplots(1,len(primary_names),figsize=(3.3*len(primary_names),3.6),squeeze=False)
for ax,name in zip(axes[0],primary_names):
    result=RESULTS[(selected_seed,name)]
    sns.heatmap(confusion_matrix(result['y'],result['probs'].argmax(1),labels=[0,1,2]),annot=True,fmt='d',cmap='Blues',cbar=False,ax=ax,xticklabels=LABELS,yticklabels=LABELS)
    ax.set(title=short_names[name],xlabel='Predicted',ylabel='True')
fig.suptitle('Confusion matrices : validation-selected seed'); plt.show()
per_class=[]; per_sweet=[]
for name in primary_names:
    result=RESULTS[(selected_seed,name)]; pred=result['probs'].argmax(1)
    p,r,f,n=precision_recall_fscore_support(result['y'],pred,labels=[0,1,2],zero_division=0)
    for i,level in enumerate(LABELS): per_class.append(dict(Model=short_names[name],Intensity=level,Precision=p[i],Recall=r[i],F1=f[i],Support=n[i]))
    frame=test_loader.dataset.frame.iloc[result['indices']]
    for sweet in SWEETS:
        mask=(frame.sweet==sweet).to_numpy()
        per_sweet.append(dict(Model=short_names[name],Sweet=sweet,N=int(mask.sum()),**metrics(result['y'][mask],result['probs'][mask])))
per_class=pd.DataFrame(per_class); per_sweet=pd.DataFrame(per_sweet)
fig,axes=plt.subplots(1,3,figsize=(15,4))
for ax,metric in zip(axes,['Precision','Recall','F1']):
    sns.barplot(data=per_class,x='Intensity',y=metric,hue='Model',order=list(LABELS),ax=ax,errorbar=None); ax.set_title('Per-class '+metric); ax.set_ylim(0,1.05)
plt.show()
fig,ax=plt.subplots(figsize=(11,4)); sns.barplot(data=per_sweet,x='Sweet',y='Macro F1',hue='Model',ax=ax,errorbar=None)
ax.set_title('Per-sweet macro F1 : validation-selected seed'); plt.show()
display(per_sweet)

## 10 : Fusion gates, confidence, calibration and embeddings
Reliability diagrams use top-label confidence with 10 fixed bins. ECE weights the absolute confidence/accuracy gap by bin count; empty bins are omitted. No test-set temperature fitting is performed. Confidence distributions and gates describe model behavior, not validated chemical certainty. PCA is a descriptive projection of held-out embeddings and is never used for classification or model selection.

In [ ]:
r=selected_result; y=r['y']; probs=r['probs']; pred=probs.argmax(1); confidence=probs.max(1)
frame=test_loader.dataset.frame.iloc[r['indices']].reset_index(drop=True)
if 'gates' in r:
    gates=pd.DataFrame(r['gates'],columns=['RGB','HSV','LAB']); gates['Sweet']=frame.sweet.to_numpy()
    gate_long=gates.melt(id_vars='Sweet',var_name='Colour space',value_name='Weight')
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    sns.boxplot(data=gate_long,x='Colour space',y='Weight',ax=axes[0]); axes[0].set_title('Overall sample-dependent gates')
    sns.boxplot(data=gate_long,x='Sweet',y='Weight',hue='Colour space',ax=axes[1]); axes[1].set_title('Fusion gates by sweet')
    plt.show()
fig,axes=plt.subplots(1,2,figsize=(12,4))
sns.histplot(x=confidence,hue=np.where(pred==y,'Correct','Incorrect'),bins=np.linspace(0,1,16),multiple='layer',ax=axes[0])
axes[0].set(title='CS-MON prediction confidence',xlabel='Maximum class probability')
for name in primary_names:
    result=RESULTS[(selected_seed,name)]; ece,bins=calibration(result['y'],result['probs'])
    axes[1].plot(bins.confidence,bins.accuracy,'o-',label=f'{short_names[name]} ECE={ece:.3f}')
axes[1].plot([0,1],[0,1],'k--',alpha=.5); axes[1].set(xlim=(0,1),ylim=(0,1),title='Reliability diagram',xlabel='Mean confidence',ylabel='Observed accuracy'); axes[1].legend(fontsize=8)
plt.show()
if CONFIG['run_embeddings'] and len(r['features'])>=3:
    embedding=PCA(n_components=2,random_state=CONFIG['seed']).fit_transform(r['features'])
    fig,ax=plt.subplots(figsize=(8,5))
    sns.scatterplot(x=embedding[:,0],y=embedding[:,1],hue=[list(LABELS)[i] for i in y],style=frame.sweet,ax=ax,alpha=.7,hue_order=list(LABELS))
    ax.set(title='Held-out VMamba embeddings : descriptive PCA',xlabel='PC 1',ylabel='PC 2'); plt.show()

## 11 : Integrated Gradients on the actual VMamba model
Attribute the fixed predicted class probability to raw RGB input through fusion, colour conversions and SS2D. Use trapezoidal integration from a neutral RGB 0.5 baseline. This avoids assumptions about CNN Grad-CAM layers or feature layouts. Disable mixed precision for attribution. Report completeness residual: attribution sum should approach score(image) - score(baseline) as steps increase. A large residual means the numerical approximation needs more steps. Zero/constant attribution and nonfinite gradients are errors. The baseline is a methodological choice, not a physical no-colour control.

Examples are chosen after evaluation for illustration: a correct example from each class where available, and an incorrect prediction where present. These selections never feed back into training.

In [ ]:
def integrated_gradients(model,x,target,steps):
    model.eval(); baseline=torch.full_like(x,.5); difference=x-baseline; gradient_sum=torch.zeros_like(x)
    assert steps>=2
    for i,alpha in enumerate(torch.linspace(0,1,steps+1,device=x.device)):
        point=(baseline+alpha*difference).detach().requires_grad_(True)
        score=class_probabilities(model(point),model.ordinal)[0,target]
        grad=torch.autograd.grad(score,point)[0]
        assert torch.isfinite(grad).all(), 'Attribution gradients are nonfinite.'
        gradient_sum+=grad*(.5 if i in (0,steps) else 1.)
    attribution=difference*gradient_sum/steps
    with torch.no_grad():
        delta=(class_probabilities(model(x),model.ordinal)[0,target]-class_probabilities(model(baseline),model.ordinal)[0,target]).item()
    residual=float(attribution.sum().detach())-delta
    assert attribution.abs().sum()>0, 'No attribution signal.'
    return attribution.detach(),residual,delta

if CONFIG['run_explainability']:
    examples=[]
    for label in range(3):
        candidates=np.flatnonzero((y==label)&(pred==y))
        if len(candidates): examples.append(int(candidates[0]))
        else: print('No correct',list(LABELS)[label],'prediction is available for attribution.')
    wrong=np.flatnonzero(pred!=y)
    if len(wrong): examples.append(int(wrong[0]))
    else: print('No incorrect predictions are available for attribution.')
    if examples:
        fig,axes=plt.subplots(len(examples),3,figsize=(11,3.5*len(examples)),squeeze=False)
        for row,i in enumerate(examples):
            x=test_loader.dataset[int(r['indices'][i])][0][None].to(DEVICE)
            attr,residual,delta=integrated_gradients(selected_model,x,int(pred[i]),CONFIG['ig_steps'])
            heat=attr.abs().sum(1)[0].cpu().numpy(); heat=heat/(np.percentile(heat,99)+1e-12); heat=np.clip(heat,0,1)
            image=x[0].permute(1,2,0).cpu().numpy()
            axes[row,0].imshow(image); axes[row,1].imshow(heat,cmap='magma',vmin=0,vmax=1)
            axes[row,2].imshow(image); axes[row,2].imshow(heat,cmap='magma',alpha=.5,vmin=0,vmax=1)
            axes[row,0].set_title(f'True {list(LABELS)[y[i]]} : predicted {list(LABELS)[pred[i]]}\nConfidence {confidence[i]:.3f}')
            axes[row,1].set_title('Absolute Integrated Gradients')
            axes[row,2].set_title(f'Overlay : completeness residual {residual:.3g}\nScore difference {delta:.3g}')
            for ax in axes[row]: ax.axis('off')
        plt.show()

## 12 : Error analysis and supported robustness slices
Show correct, adjacent ordinal error (distance 1) and severe error (distance 2) examples only when they exist. Device/lighting slice metrics are conditional observations within this split, not evidence of unseen-device generalization. Missing session/batch metadata is reported explicitly. No potentially label-changing colour perturbations are treated as ground truth preserving robustness tests.

In [ ]:
error_groups={'Correct':np.flatnonzero(pred==y),'Adjacent error':np.flatnonzero(np.abs(pred-y)==1),'Severe ordinal error':np.flatnonzero(np.abs(pred-y)==2)}
fig,axes=plt.subplots(3,3,figsize=(10,10))
for row,(title,indices) in enumerate(error_groups.items()):
    for col,ax in enumerate(axes[row]):
        if col<len(indices):
            i=int(indices[col]); image=test_loader.dataset[int(r['indices'][i])][0].permute(1,2,0).numpy()
            ax.imshow(image); ax.set_title(f'{title}\n{list(LABELS)[y[i]]} : {list(LABELS)[pred[i]]} : {confidence[i]:.2f}')
        else: ax.text(.5,.5,'No further examples',ha='center')
        ax.axis('off')
plt.show()
if CONFIG['run_robustness'] or CONFIG['mode']=='full' and CONFIG['full_run_robustness']:
    slices=[]
    for column in ['device','lighting','capture_session_id','preparation_batch_id']:
        if column not in frame or not frame[column].astype(str).str.strip().ne('').any():
            print(column+': unavailable; corresponding generalization cannot be directly tested.'); continue
        for value in sorted(frame[column].unique()):
            if str(value).strip()=='': continue
            mask=(frame[column]==value).to_numpy()
            slices.append(dict(Metadata=column,Value=value,N=int(mask.sum()),Classes=len(np.unique(y[mask])),**metrics(y[mask],probs[mask])))
    if slices:
        robustness=pd.DataFrame(slices); display(robustness)
        fig,ax=plt.subplots(figsize=(11,4)); sns.barplot(data=robustness,x='Value',y='Macro F1',hue='Metadata',ax=ax,errorbar=None)
        ax.tick_params(axis='x',rotation=30); ax.set_title('Observed metadata slices : CS-MON'); plt.show()

## 13 : Ablation results and measured interpretation
Ablations are optional in full mode and skipped in quick mode. No trained configuration is duplicated within a seed. A3/A4 alias the primary result only if fusion, objective and augmentation all match. Comparisons on the held-out test set are final descriptive analyses, not a tuning signal. Multiple seeds do not remove dependence or label-provenance limitations.

In [ ]:
ablation_results=all_results[all_results.Model.str.match(r'^A[0-4]$')]
if len(ablation_results):
    display(ablation_results.groupby('Model').agg({metric:['mean','std'] for metric in ['Accuracy','Macro F1','Ordinal MAE','QWK']}))
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    for ax,metric in zip(axes,['Macro F1','Ordinal MAE']):
        means=ablation_results.groupby('Model')[metric].mean(); std=ablation_results.groupby('Model')[metric].std().fillna(0)
        ax.bar(means.index,means,yerr=std,capsize=3); ax.set(title=metric,xlabel='Ablation',ylabel=metric)
    plt.show()
else: print('Ablations skipped by the predeclared execution mode/configuration.')
comparison_bars(['Accuracy','Macro F1','Ordinal MAE','QWK'],'Final model comparison : '+('preliminary quick run' if CONFIG['mode']=='quick' else 'full experiment'))
if CONFIG['run_baselines']:
    baseline_f1=comparison.loc[BASELINE_NAMES,'Macro F1']; strongest=baseline_f1.idxmax()
    difference=float(comparison.loc[CSMON,'Macro F1']-baseline_f1.max())
    print(f'CS-MON macro F1 minus the strongest measured baseline ({strongest}): {difference:+.4f} ({difference*100:+.2f} percentage points).')
    if difference<=0: print('CS-MON did not exceed the strongest baseline on this metric. Inspect validation convergence, data independence, adapter behavior and sample size before predeclaring further experiments.')
print('No significance claim is made from a single run. Device/lighting slices are not unseen-domain holdouts.')
print('Labels require documented preparation/reference measurements; image predictions and attributions do not identify dyes, quantify chemical concentrations or establish food safety.')

## 14 : Save one reconstructable, validation-selected CS-MON checkpoint
Write exactly `outputs/best_csmon.pt` once. The selected state contains the fine-tuned backbone, learned adapter, ordinal head and normalization. Include architecture, source/checkpoint provenance, manifest hash, seed, selected validation epoch/metric, colour ranges and versions. Optimizer states and baseline weights are excluded. Reload without a new ImageNet initialization and verify strict loading plus identical predictions on a validation batch. Reconstruction still requires the pinned official source and software dependencies recorded here, not the original ImageNet checkpoint weights.

In [ ]:
PREPROCESSING={'image_size':CONFIG['image_size'],'orientation':'EXIF transpose','roi':'whole image',
 'resize':'Pillow LANCZOS square','rgb':'sRGB [0,1]','hsv':'Kornia H/(2*pi), S,V [0,1]',
 'lab':'D65 L/50-1,a/128,b/128','fusion_rgb_hsv':'2*x-1',
 'backbone_normalization':{'mean':[.485,.456,.406],'std':[.229,.224,.225]},
 'augmentation':'horizontal/vertical flips' if selected_run['spec']['augment'] else 'none'}

def restore_csmon(checkpoint_path):
    payload=torch.load(checkpoint_path,map_location='cpu',weights_only=False)
    assert payload['class_mapping']==LABELS
    assert payload['vmamba_provenance']['commit']==VMAMBA_PROVENANCE['commit']
    assert payload['vmamba_provenance']['source_sha256']==VMAMBA_PROVENANCE['source_sha256']
    assert payload['architecture']['vmamba_kwargs']==VMAMBA_KWARGS
    restored=construct(payload['architecture']['spec'],CSMON,pretrained=False).to(DEVICE)
    restored.load_state_dict(payload['state_dict'],strict=True); restored.eval()
    return restored,payload

if CONFIG['save_best_model']:
    assert selected_run['backbone_updated'] and selected_run['selected']['stage']!='head'
    payload={'format_version':1,'model_name':CSMON,'state_dict':selected_run['state'],
       'architecture':{'vmamba_kwargs':copy.deepcopy(VMAMBA_KWARGS),'spec':selected_run['spec'],'fusion_channels':12,'head':'ordered-cutpoint CORAL'},
       'config':copy.deepcopy(CONFIG),'preprocessing':PREPROCESSING,'class_mapping':LABELS,
       'selected_epoch':selected_run['selected']['epoch'],'selection_metric':'validation macro F1; validation loss tie break',
       'selection':selected_run['selected'],'seed':SELECTED_KEY[0],'manifest_sha256':MANIFEST_SHA256,
       'versions':VERSIONS,'hardware':HARDWARE,'vmamba_provenance':VMAMBA_PROVENANCE,
       'label_reference_documentation':CONFIG['label_reference_documentation'] or 'UNVERIFIED: supplied CSV labels only',
       'split_scope':manifest.split_scope.unique().tolist()}
    output=ROOT/'outputs'; output.mkdir(exist_ok=True); path=output/'best_csmon.pt'
    torch.save(payload,path)
    restored,saved=restore_csmon(path)
    x=next(iter(make_loaders(CONFIG['seed'],False)['validation']))[0][:1].to(DEVICE)
    with torch.no_grad():
        torch.testing.assert_close(restored(x),selected_model(x),rtol=1e-5,atol=1e-6)
    del restored; gc.collect()
    print('Saved and reloaded:',path)
    print('Pinned VMamba source:',saved['vmamba_provenance'])

## Implementation verification performed in this workspace

The preprocessing notebook completed on all 1,944 original images and generated the audited split manifest. File hashes and split constraints were revalidated in the Linux GPU environment.

**GPU environment verified:** Ubuntu 22.04.5 / Python 3.10.12 / PyTorch 2.2.2+cu121 / CUDA 12.1 / Triton 2.2.0 / official Mamba selective scan 2.2.4, using the GTX 1650 Ti (4 GB). The complete notebook preflight passed on real 224 x 224 images, including strict authentic pretrained loading, colour conversions, identity fusion, gradients through the frozen backbone, a real VMamba parameter update, valid ordinal probabilities and all four pretrained baseline outputs. The shared training loop passed a separate **debug-only** staged CS-MON run with CUDA AMP: 6 actual training images, 3 validation images and 3 evaluation images, selected from the audited splits. This verified accumulation, validation selection and the actual GPU scan backward path. Measured peak allocated GPU memory in that small check was approximately 0.78 GB. No debug checkpoint or debug performance table is delivered.

The registered **csmon-gpu** Jupyter kernel was launched separately and successfully verified its interpreter and CUDA device. Installed dependencies passed the environment consistency check. Earlier CPU checks also verified algebraic ordinal assertions, metrics, serialization and the experiment-plan reuse logic.

**Not executed:** full-data quick/full research training, research model comparison plots, ablation training, final-model Integrated Gradients, or final research checkpoint saving/reload. Run All in the verified WSL GPU kernel starts those configured experiment steps. No `best_csmon.pt` has been generated by environment setup. Label provenance and physical dependence metadata remain explicit scientific limitations.
